비전환고객

In [ ]:
# ============================================================
# viewed 데이터 분리
# ============================================================
viewed = (
    df2[df2['event'] == 'offer viewed'][
        ['customer_id', 'offer_id', 'time']
    ]
    .copy()
    .rename(columns={'time': 'time_viewed'})
)

# 정렬 (안정성)
viewed = viewed.sort_values(
    ["customer_id", "offer_id", "time_viewed"]
).reset_index(drop=True)


# ============================================================
# 각 received 기준으로 viewed 여부 판단
# ============================================================
view_flag_list = []

for idx, row in rc.iterrows():
    
    cust = row['customer_id']
    offer = row['offer_id']
    r_time = row['time_received']
    duration = row['duration'] * 24  # 시간 단위 맞추기
    
    # 해당 고객 + 오퍼의 viewed만 추출
    viewed_group = viewed[
        (viewed['customer_id'] == cust) &
        (viewed['offer_id'] == offer)
    ]
    
    # 유효 기간 내 viewed 존재 여부
    has_viewed = (
        (viewed_group['time_viewed'] >= r_time) &
        (viewed_group['time_viewed'] <= r_time + duration)
    ).any()
    
    view_flag_list.append(int(has_viewed))

# 컬럼 추가
rc['has_viewed'] = view_flag_list


# ============================================================
# 🔥 최종 상태 분류
# ============================================================
def classify_status(row):
    if row['converted_final'] == 1:
        return 'converted'
    elif row['has_viewed'] == 0:
        return 'not_viewed'
    else:
        return 'viewed_not_converted'

rc['status'] = rc.apply(classify_status, axis=1)

중도이탈자는 전체의 31%, 전환자 수 대비 약 59% 수준으로 나타났다(viewed->completed로 유도하는 방향 제시)

In [ ]:
print("=" * 80)
print("상태별 개체수 + 비율")
print("=" * 80)

status_summary = (
    rc['status']
    .value_counts()
    .to_frame(name='count')  # 개체수
)

status_summary['ratio(%)'] = (
    status_summary['count'] / len(rc) * 100
).round(2)

display(status_summary)

In [ ]:
print("=" * 80)
print("퍼널 구조")
print("=" * 80)

total = len(rc)
viewed = rc['has_viewed'].sum()
converted = rc['converted_final'].sum()

print(f"received: {total}")
print(f"viewed: {viewed} ({viewed/total*100:.2f}%)")
print(f"converted: {converted} ({converted/total*100:.2f}%)")

In [ ]:
plt.figure()

offer_status = (
    rc.groupby(['offer_type', 'status'])
      .size()
      .unstack(fill_value=0)
)

offer_status.plot(kind='bar', stacked=True)

plt.title('Offer Type별 상태 분포')
plt.xlabel('Offer Type')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(title='Status')

plt.show()

In [ ]:
# 채널 펼치기
channel_cols = ['web', 'email', 'mobile', 'social']

channel_df = rc.copy()

channel_long = channel_df.melt(
    id_vars=['status'],
    value_vars=channel_cols,
    var_name='channel',
    value_name='used'
)

# 채널 사용된 것만
channel_long = channel_long[channel_long['used'] == 1]

# 집계
channel_status = (
    channel_long.groupby(['channel', 'status'])
    .size()
    .unstack(fill_value=0)
)

# 시각화
channel_status.plot(kind='bar', stacked=True)

plt.title('채널별 상태 분포')
plt.xlabel('Channel')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.show()

-----------------------
offer 효과(rfm 소비금액, 구매 빈도)

In [ ]:
# 1. 전환 고객 필터
converted_rc = rc[rc['converted_final'] == 1].copy()

# 2. 거래 데이터
transactions = df[df['event'] == 'transaction'][
    ['customer_id', 'time', 'amount']
].copy()

# 3. 고객 + 거래 merge
merged = converted_rc.merge(
    transactions,
    on='customer_id',
    how='left'
)

# 4. before / after (±7일 기준)
merged['before_flag'] = (
    (merged['time'] >= merged['time_received'] - 7*24) &
    (merged['time'] < merged['time_received'])
)

merged['after_flag'] = (
    (merged['time'] >= merged['time_received']) &
    (merged['time'] <= merged['time_received'] + 7*24)
)

# 5. before / after 나누기
before = merged[merged['before_flag']]
after = merged[merged['after_flag']]

# 6. 집계
before_agg = before.groupby('customer_id').agg(
    before_count=('amount', 'count'),
    before_amount=('amount', 'sum')
)

after_agg = after.groupby('customer_id').agg(
    after_count=('amount', 'count'),
    after_amount=('amount', 'sum')
)

# 7. 결합
compare_final = before_agg.join(after_agg, how='outer').fillna(0)

# 8. 평균 비교 (최종 결과)
print("=" * 80)
print("전 / 후 평균 비교 (±7일 기준)")
print("=" * 80)

print("▶ 구매 횟수")
print("Before 평균:", compare_final['before_count'].mean())
print("After 평균:", compare_final['after_count'].mean())

print("\n▶ 구매 금액")
print("Before 평균:", compare_final['before_amount'].mean())
print("After 평균:", compare_final['after_amount'].mean())